# Registro de experimentos del proyecto

Atik Santellán

**Proyecto:** Predicción de depreciación de al menos 40 % del valor de mercado de futbolistas.

Este notebook reutiliza la preparación y la división temporal de `pipeline_proyecto.py` y las configuraciones de la Semana 10. Se registran **cuatro corridas** en MLflow: línea base, regresión logística, gradient boosting y red neuronal.

**Antes de ejecutar:** deja este notebook al lado de `pipeline_proyecto.py` y `features_proyecto.csv`. Inicia desde la carpeta del notebook. Si este experimento ya tiene corridas de prueba en MLflow, elimínalas antes de la ejecución definitiva, reinicia el kernel y usa **Run All** una sola vez.

**Criterio principal de selección:** `balanced_acc_prueba` (igual al utilizado en `modelos_proyecto.py`). Se registran también F1, precision y recall de la clase positiva.

## 1. Importaciones y conexión a MLflow

Se utiliza el archivo `mlflow.db` local de la carpeta actual, como en los dos ejercicios anteriores. Con MLflow 3.18 y Python 3.14, serializaremos los modelos **propios y confiables** con `cloudpickle` para evitar la lista de tipos de `skops` que varía entre estimadores. Nunca cargues artefactos pickle/cloudpickle de procedencia desconocida.

In [1]:
from pathlib import Path
from importlib import import_module
import hashlib
import time

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import sklearn
from mlflow.tracking import MlflowClient
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight

EXPERIMENTO = "depreciacion-futbolistas"
ARCHIVO_DATOS = Path("features_proyecto.csv")
ARCHIVO_SALIDA = Path("corridas_proyecto.csv")

assert ARCHIVO_DATOS.is_file(), f"No existe {ARCHIVO_DATOS} en {Path.cwd()}"
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment(EXPERIMENTO)

print("MLflow:", mlflow.__version__)
print("scikit-learn:", sklearn.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())

2026/11/08 01:21:10 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/08 01:21:10 INFO mlflow.store.db.utils: Updating database tables
2026/11/08 01:21:12 INFO mlflow.tracking.fluent: Experiment with name 'depreciacion-futbolistas' does not exist. Creating a new experiment.


MLflow: 3.18.0
scikit-learn: 1.9.1
Tracking URI: sqlite:///mlflow.db


## 2. Importar división y preparación de la Semana 9

Este archivo original contiene variables ya definidas (`X_ent`, `X_prueba`, `y_ent`, `y_prueba`, `fecha_corte`, `separacion`) y un `ColumnTransformer` llamado `preparacion`. **No contiene funciones `dividir()` y `preparacion()`**. Por eso importamos esas variables y usamos `clone()` para construir un transformador independiente y sin ajustar para cada modelo. El módulo original ejecuta una preparación de diagnóstico al importarse; **no reutilizamos ese transformador ajustado** dentro de las corridas.

In [2]:
# Acepta el nombre exigido por la entrega y el prefijo de los archivos compartidos.
if Path("pipeline_proyecto.py").exists():
    fuente = import_module("pipeline_proyecto")
elif Path("ejercicio3_pipeline_proyecto.py").exists():
    fuente = import_module("ejercicio3_pipeline_proyecto")
else:
    raise FileNotFoundError("Coloca pipeline_proyecto.py junto al notebook")

X_ent, X_prueba = fuente.X_ent, fuente.X_prueba
y_ent, y_prueba = fuente.y_ent, fuente.y_prueba
fecha_corte, separacion = fuente.fecha_corte, fuente.separacion

# El archivo se lee por bloques para evitar cargarlo nuevamente solo por el hash.
def sha256_archivo(ruta):
    sha = hashlib.sha256()
    with open(ruta, "rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            sha.update(bloque)
    return sha.hexdigest()

huella_datos = sha256_archivo(ARCHIVO_DATOS)

print("Huella SHA-256:", huella_datos)
print("Fecha de corte:", fecha_corte.date())
print("Última fecha permitida para entrenamiento:", (fecha_corte - separacion).date())
print("Entrenamiento:", len(X_ent), "Prueba:", len(X_prueba))
print("Separación temporal (días):", separacion.days)

Fecha de corte: 2023-02-09
Ultima fecha permitida para entrenamiento: 2021-12-06
Filas de entrenamiento: 302189
Filas de prueba: 90018
Filas reservadas entre ambos periodos: 57871

Entrenamiento: (302189, 82)
Prueba: (90018, 82)

Columnas despues de preparar:
['numericas__EDAD_ANIOS', 'numericas__HEIGHT_IN_CM', 'numericas__MARKET_VALUE_IN_EUR', 'numericas__VALOR_PREVIO_12M_EUR', 'numericas__VARIACION_PREVIA_12M_PCT', 'numericas__DESFASE_PREVIO_12M_DIAS', 'numericas__PARTIDOS_180D', 'numericas__MINUTES_PLAYED_180D', 'numericas__GOALS_180D', 'numericas__ASSISTS_180D', 'numericas__PARTIDOS_365D', 'numericas__MINUTES_PLAYED_365D', 'numericas__GOALS_365D', 'numericas__ASSISTS_365D', 'numericas__CAIDA_PREVIA_20', 'numericas__missingindicator_HEIGHT_IN_CM', 'numericas__missingindicator_VALOR_PREVIO_12M_EUR', 'numericas__missingindicator_VARIACION_PREVIA_12M_PCT', 'numericas__missingindicator_DESFASE_PREVIO_12M_DIAS', 'numericas__missingindicator_MINUTES_PLAYED_180D', 'numericas__missingindica

## 3. Los cuatro modelos originales de la Semana 10

- **Línea base:** `DummyClassifier(strategy="most_frequent")`.
- **Modelo lineal:** `LogisticRegression(C=1.0, class_weight="balanced", max_iter=500, random_state=42)`.
- **Ensamblaje:** `HistGradientBoostingClassifier` con los parámetros originales.
- **Red neuronal:** `MLPClassifier` con una capa de 64 neuronas, entrenamiento de hasta 200 iteraciones, batch de 512, parada anticipada y **pesos de clase** calculados a partir de `y_ent`.

El LLM de la Semana 10 no se incluye porque aquí se exige registrar cuatro modelos entrenados con el pipeline de scikit-learn.

In [3]:
modelos = [
    ("linea base", DummyClassifier(strategy="most_frequent"), False),
    ("regresion logistica", LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=500,
        random_state=42,
    ), False),
    ("gradient boosting", HistGradientBoostingClassifier(
        max_iter=100,
        max_leaf_nodes=15,
        learning_rate=0.08,
        l2_regularization=1.0,
        class_weight="balanced",
        random_state=42,
    ), False),
    ("red neuronal", MLPClassifier(
        hidden_layer_sizes=(64,),
        max_iter=200,
        batch_size=512,
        early_stopping=True,
        n_iter_no_change=10,
        random_state=42,
    ), True),
]

assert len(modelos) == 4
print([nombre for nombre, _, _ in modelos])

['linea base', 'regresion logistica', 'gradient boosting', 'red neuronal']


## 4. Registrar las cuatro corridas

En cada corrida se registran los hiperparámetros del estimador, configuración de división, huella SHA-256 del archivo, métricas de entrenamiento y prueba, segundos de entrenamiento y **pipeline completo ajustado**. El ajuste del transformador ocurre únicamente sobre `X_ent`.

In [4]:
def metricas_clasificacion(y_real, predicciones, sufijo):
    return {
        f"balanced_acc_{sufijo}": balanced_accuracy_score(y_real, predicciones),
        f"f1_{sufijo}": f1_score(y_real, predicciones, zero_division=0),
        f"precision_{sufijo}": precision_score(y_real, predicciones, zero_division=0),
        f"recall_{sufijo}": recall_score(y_real, predicciones, zero_division=0),
    }


def evaluar_y_registrar(nombre, modelo, balancear):
    # clone evita reutilizar el transformador que el módulo de la Semana 9 ajusta al importar.
    pipeline = Pipeline([
        ("preparar", clone(fuente.preparacion)),
        ("modelo", modelo),
    ])

    with mlflow.start_run(run_name=nombre) as run:
        # Los parámetros pueden ser str/int/float/bool; MLflow los almacena como texto.
        mlflow.log_params({k: str(v) for k, v in modelo.get_params().items()})
        mlflow.log_params({
            "fecha_corte": str(fecha_corte.date()),
            "ultima_fecha_entrenamiento": str((fecha_corte - separacion).date()),
            "separacion_dias": separacion.days,
            "metodo_division": "temporal_con_separacion",
            "huella_datos_sha256": huella_datos,
            "archivo_datos": ARCHIVO_DATOS.name,
            "filas_entrenamiento": len(X_ent),
            "filas_prueba": len(X_prueba),
            "pesos_muestra_balanceados": balancear,
        })
        mlflow.set_tags({
            "tipo_problema": "clasificacion_binaria",
            "metrica_principal": "balanced_acc_prueba",
            "version_mlflow": mlflow.__version__,
            "version_sklearn": sklearn.__version__,
        })

        inicio = time.perf_counter()
        if balancear:
            pesos = compute_sample_weight(class_weight="balanced", y=y_ent)
            pipeline.fit(X_ent, y_ent, modelo__sample_weight=pesos)
        else:
            pipeline.fit(X_ent, y_ent)
        segundos = time.perf_counter() - inicio

        pred_ent = pipeline.predict(X_ent)
        pred_prueba = pipeline.predict(X_prueba)
        metricas = {
            **metricas_clasificacion(y_ent, pred_ent, "entrenamiento"),
            **metricas_clasificacion(y_prueba, pred_prueba, "prueba"),
            "segundos": segundos,
        }
        mlflow.log_metrics(metricas)

        # Artefacto propio/confiable. Cloudpickle evita los problemas de skops
        # observados con Histogram Gradient Boosting y Adam en MLflow 3.18.
        mlflow.sklearn.log_model(
            pipeline,
            artifact_path="modelo",
            serialization_format="cloudpickle",
        )

        print(
            f"{nombre:20s} run_id={run.info.run_id} "
            f"balanced_acc_prueba={metricas['balanced_acc_prueba']:.4f} "
            f"F1_prueba={metricas['f1_prueba']:.4f} "
            f"segundos={segundos:.2f}"
        )
        return run.info.run_id


ids_generados = [
    evaluar_y_registrar(nombre, modelo, balancear)
    for nombre, modelo, balancear in modelos
]
assert len(ids_generados) == 4

2026/11/08 01:21:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/08 01:21:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


linea base           run_id=89dbb2111ddc41548bd1d1016e9ce316 balanced_acc_prueba=0.5000 F1_prueba=0.0000 segundos=1.42


2026/11/08 01:21:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/08 01:21:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


regresion logistica  run_id=07cc3c98ccb14d94a4c717c9af826458 balanced_acc_prueba=0.6796 F1_prueba=0.3777 segundos=7.83


2026/11/08 01:21:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/08 01:21:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


gradient boosting    run_id=82c8bc1a272445b79eef00c1d97e0c5d balanced_acc_prueba=0.6975 F1_prueba=0.3959 segundos=4.10


2026/11/08 01:21:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/08 01:21:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


red neuronal         run_id=6ac054fe3da143428931e2213cea2e14 balanced_acc_prueba=0.6792 F1_prueba=0.3792 segundos=15.10


## 5. Comparación y selección

**Antes de ejecutar esta celda en la entrega final**, el experimento debe contener exactamente las cuatro corridas que acabamos de entrenar; elimina corridas exploratorias desde la interfaz de MLflow y vuelve a ejecutar el notebook completo desde un kernel reiniciado. La selección reproduce el criterio de la Semana 10: **mayor balanced accuracy de prueba**.

In [5]:
corridas = mlflow.search_runs(experiment_names=[EXPERIMENTO])
if len(corridas) != 4:
    raise RuntimeError(
        f"Se esperaban 4 corridas y hay {len(corridas)}. "
        "Elimina las corridas de prueba, reinicia el kernel y ejecuta Run All."
    )

corridas = corridas.sort_values(
    "metrics.balanced_acc_prueba", ascending=False
).reset_index(drop=True)

columnas = [
    "tags.mlflow.runName", "metrics.balanced_acc_entrenamiento",
    "metrics.balanced_acc_prueba", "metrics.f1_prueba",
    "metrics.precision_prueba", "metrics.recall_prueba",
    "metrics.segundos", "run_id",
]
display(corridas[columnas])

mejor_run_id = corridas.loc[0, "run_id"]
mejor_nombre = corridas.loc[0, "tags.mlflow.runName"]
mejor_metrica = float(corridas.loc[0, "metrics.balanced_acc_prueba"])
print("Ganador:", mejor_nombre)
print("run_id:", mejor_run_id)
print("Balanced accuracy de prueba:", mejor_metrica)

,tags.mlflow.runName,metrics.balanced_acc_entrenamiento,metrics.balanced_acc_prueba,metrics.f1_prueba,metrics.precision_prueba,metrics.recall_prueba,metrics.segundos,run_id
0,gradient boosting,0.706452,0.697517,0.395870,0.280112,0.674692,4.097693,82c8bc1a272445b79eef00c1d97e0c5d
1,regresion logistica,0.686484,0.679583,0.377745,0.267284,0.643817,7.828137,07cc3c98ccb14d94a4c717c9af826458
2,red neuronal,0.711434,0.679167,0.379177,0.270385,0.634458,15.098498,6ac054fe3da143428931e2213cea2e14
3,linea base,0.500000,0.500000,0.000000,0.000000,0.000000,1.417755,89dbb2111ddc41548bd1d1016e9ce316


Ganador: gradient boosting
run_id: 82c8bc1a272445b79eef00c1d97e0c5d
Balanced accuracy de prueba: 0.6975168280830886


## 6. Marcar candidato final y recuperar el modelo

Se añade el tag `candidato=final` al `run_id` ganador y se recupera el pipeline serializado. La comparación se realiza sobre **exactamente `X_prueba` y `y_prueba`**; esto comprueba la recuperación del modelo, no su capacidad para generalizar a otro conjunto de prueba.

In [6]:
cliente = MlflowClient()
cliente.set_tag(mejor_run_id, "candidato", "final")

modelo_recuperado = mlflow.sklearn.load_model(
    f"runs:/{mejor_run_id}/modelo"
)
predicciones_recuperadas = modelo_recuperado.predict(X_prueba)
metrica_reproducida = balanced_accuracy_score(y_prueba, predicciones_recuperadas)

print("Balanced accuracy registrada:", mejor_metrica)
print("Balanced accuracy reproducida:", metrica_reproducida)
assert np.isclose(mejor_metrica, metrica_reproducida, rtol=0, atol=1e-12)
print("Verificación: OK")

Balanced accuracy registrada: 0.6975168280830886
Balanced accuracy reproducida: 0.6975168280830886
Verificación: OK


## 7. Exportar `corridas_proyecto.csv`

La exportación se realiza **después** de etiquetar al candidato final, de modo que el CSV incluya el campo `tags.candidato`. No se exporta manualmente una tabla construida desde las métricas de Python, sino la consulta real a MLflow.

In [7]:
corridas_exportadas = mlflow.search_runs(
    experiment_names=[EXPERIMENTO]
)
assert len(corridas_exportadas) == 4
assert (corridas_exportadas["tags.candidato"] == "final").sum() == 1

corridas_exportadas.to_csv("corridas_proyecto.csv", index=False)
print("CSV guardado:", ARCHIVO_SALIDA.resolve())
print("Número de corridas:", len(corridas_exportadas))

CSV guardado: /home/satt/icebka/usfq/dmining/ciencia-de-datos/laboratorios/semana-08/data-science-end-to-end/laboratorios/semana-11-sesion-01/desarrollo/ejercicio3/corridas_proyecto.csv
Número de corridas: 4
